# SuperKernel 融合优化功能

## 1. 功能简介

SuperKernel 是一种算子二进制融合技术。它不改写算子源码，而是在已编译的 Kernel 基础上，将多个可融合任务组织为一个超级 Kernel，通过减少任务下发、调度等待和算子头部开销来提升资源利用率。

SuperKernel 并不保证所有模型都能获得收益：融合范围、算子依赖、同步行为和资源占用都会影响最终效果，因此必须结合 Profiling 和精度结果评估。

**实现原理**：基于 aclgraph 捕获后的 Model，SuperKernel 模块识别可被融合的连续 Task，将其合并为一个 SuperKernel Task 并替换原有 Task，通过 Runtime 接口更新 Model 后基于新 Model 执行。

开启 SuperKernel 融合优化后，系统自动识别图内可被融合的算子，在 SuperKernel 内以子函数调用的方式依次执行。同时提供标定 SuperKernel 范围的能力，支持用户根据实际业务需求对融合范围内的算子进行标记和优化配置。

## 2. 使用约束

- 本功能支持如下产品：Atlas A3 训练系列产品/Atlas A3 推理系列产品、Atlas A2 训练系列产品/Atlas A2 推理系列产品。
- SuperKernel 融合会按照网络中算子的顺序依次判断是否可被融合。当识别到不可融合的算子时，系统会生成第一段 SuperKernel，并自动跳过该算子进行第二段 SuperKernel 融合。
- 目前支持 SuperKernel 融合的通信类算子包括 AllReduce、ReduceScatter、AllGather、AlltoAll。
- 开启 SuperKernel 融合优化时，**需要同时开启** [静态 Kernel 编译功能](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph%5Fex/basic/static%5Fkernel%5Fcompile.md)。
  > 同一进程内存在多个 Model 时，若不同 Model 中存在输入输出 Shape 完全相同的算子，但 SuperKernel 相关选项配置不一致，后续相同 Shape 算子可能复用前一次静态 Kernel 编译产物，导致 SuperKernel 融合不符合预期。建议这些 Model 保持配置一致，或仅在需要 SuperKernel 的 Model 上开启静态 Kernel 编译。
- 开启 SuperKernel 融合优化后，[算子 Data Dump 功能](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph%5Fex/dfx/data%5Fdump.md)将会失效。
- 运行本示例需要 CANN 环境中包含 `libascendsk.so` 动态库，若缺失请参考 CANN 安装文档确认环境配置。

## 3. 使用方法

### 3.1 标定 SuperKernel 范围（可选）

使用 `super_kernel_scope_begin` 和 `super_kernel_scope_end` 标定融合范围，语句块内的算子将被融合为一个 SuperKernel：

```python
torch.npu.super_kernel_scope_begin(scope_name: str)
# 待融合的算子操作
torch.npu.super_kernel_scope_end(scope_name: str)
```

- `scope_name`：表示该范围内算子融合后的 SuperKernel 名称，相同的 `scope_name` 代表相同的融合范围。若传入 `None`，则该范围内的算子不进行 SuperKernel 融合。
- `scope_name` 名称不超过 255 字节，全局支持的 `scope_name`（不同）数量不超过 1024 个。
- 标定范围不代表 SuperKernel 的融合结果，系统仍会按依赖关系自动判断。

> 当多个标定范围有交集时，按照标记集对算子进行划分。标记为 `None` 的范围具备最高优先级，被标记的算子不进行融合。

### 3.2 通过 options 配置开启

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr><th align="left">参数名</th><th align="left">说明</th></tr>
  <tr><td align="left"><code>super_kernel_optimize</code></td><td align="left">布尔类型，是否开启 SuperKernel 融合优化。<code>False</code>（默认）：关闭；<code>True</code>：开启。</td></tr>
  <tr><td align="left"><code>super_kernel_optimize_options</code></td><td align="left">字典类型，SuperKernel 融合优化参数。</td></tr>
  <tr><td align="left"><code>super_kernel_debug_options</code></td><td align="left">字典类型，SuperKernel 融合调试参数。<strong>建议仅在功能调试时使用。</strong></td></tr>
</table>
<div style="clear: both;"></div>


### 3.3 super_kernel_optimize_options 常用参数

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr><th align="left">参数名</th><th align="left">说明</th></tr>
  <tr><td align="left"><code>dcci_before_kernel_start</code></td><td align="left">在 SuperKernel 调用指定算子前插入 DataCacheCleanAndInvalid 指令刷新缓存。配置格式：<code>[".*op_type.*"]</code>。</td></tr>
  <tr><td align="left"><code>dcci_after_kernel_end</code></td><td align="left">在 SuperKernel 调用指定算子后插入 DataCacheCleanAndInvalid 指令刷新缓存。配置格式：<code>[".*op_type.*"]</code>。</td></tr>
  <tr><td align="left"><code>dcci_disable_on_kernel</code></td><td align="left">在 SuperKernel 调用指定算子前后不插入任何 DataCacheCleanAndInvalid 指令。优先级低于前两者。</td></tr>
  <tr><td align="left"><code>auto_op_parallel</code></td><td align="left">多流场景下，是否使能 Cube/Vector 自动并行排布。<code>0</code>（默认）：不使能；<code>1</code>：使能。</td></tr>
  <tr><td align="left"><code>early_start</code></td><td align="left">是否启用 SuperKernel Early-Start 优化。<code>0</code>（默认）：关闭；<code>1</code>：开启，使算子头部 Scalar 指令提前执行。</td></tr>
  <tr><td align="left"><code>aggressive_opt_strategies</code></td><td align="left">字典类型，开启激进融合策略。包含 <code>task_breaker_bypass</code> 和 <code>value_breaker_bypass</code> 子项。</td></tr>
</table>
<div style="clear: both;"></div>


### 3.4 super_kernel_debug_options 参数

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr><th align="left">参数名</th><th align="left">说明</th></tr>
  <tr><td align="left"><code>debug_sync_all</code></td><td align="left">在 SuperKernel 内每个算子间自动插入 SyncAll 全核同步指令，辅助定界同步问题。<code>0</code>（默认）：关闭；<code>1</code>：开启。</td></tr>
  <tr><td align="left"><code>debug_op_exec_trace</code></td><td align="left">启用算子执行轨迹追踪，异常时在 plog 中打印算子执行进度信息。<code>0</code>（默认）：关闭；<code>1</code>：开启。</td></tr>
  <tr><td align="left"><code>debug_cross_core_sync_check</code></td><td align="left">启用子算子 Cube&amp;Vector 核间同步校验功能。<code>0</code>（默认）：关闭；<code>1</code>：开启。</td></tr>
  <tr><td align="left"><code>debug_per_op_max_core_num</code></td><td align="left">启用单算子满核验证模式，融合范围内每个算子独立成为 SuperKernel 并以最大核数运行。<code>0</code>（默认）：关闭；<code>1</code>：开启。</td></tr>
</table>
<div style="clear: both;"></div>


## 4. 使用示例

下面的示例展示如何使用 SuperKernel 融合优化。通过 `super_kernel_scope_begin` 和 `super_kernel_scope_end` 标定融合范围，同时开启静态 Kernel 编译和 SuperKernel 融合优化。

In [ ]:
import torch
import torch_npu


class SuperKernelModel(torch.nn.Module):
    def forward(self, x, y, weight):
        # begin/end 仅标记候选融合范围，不保证其中所有算子最终都能融合。
        torch.npu.super_kernel_scope_begin("pointwise_block")
        hidden = torch.mm(x, weight)
        hidden = torch.relu(hidden)
        hidden = hidden + y
        torch.npu.super_kernel_scope_end("pointwise_block")
        return hidden


model = SuperKernelModel().npu()
# SuperKernel 依赖静态 Kernel 编译，两项能力需同时开启。
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={
        "static_kernel_compile": True,
        "super_kernel_optimize": True,
        "super_kernel_optimize_options": {},
        "super_kernel_debug_options": {},
    },
    dynamic=False,
    fullgraph=True,
)
x = torch.randn(32, 64, dtype=torch.float16).npu()
y = torch.randn(32, 64, dtype=torch.float16).npu()
weight = torch.randn(64, 64, dtype=torch.float16).npu()
# 首次执行完成编译后，可用 Profiler 确认实际融合结果和性能。
out = compiled(x, y, weight)
torch.npu.synchronize()
print("output:", tuple(out.shape))


## 5. 调试与调优

如需确认 SuperKernel 融合结果、定位执行异常或分析性能无收益、劣化等问题，可借助以下方式：

- 使用 `super_kernel_debug_options` 中的调试选项（如 `debug_op_exec_trace`、`debug_cross_core_sync_check` 等）定位问题。
- 使用 Profiling 工具确认融合结果和性能表现。
- 详细调试调优方法请参考 [SuperKernel 调试调优方法](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/appendix/cases/superkernel%5Fcases.md#npugraph%5Fex%E5%90%8E%E7%AB%AF)。

> **注意**：`super_kernel_debug_options` 中的选项均会影响 SuperKernel 融合效果或运行性能，建议仅在功能调试时使用。

## 6. 课后练习

### 一、单选题

（1）【单选题】SuperKernel 的核心优化方式是？
- A. 重写所有算子源码
- B. 将可融合的多个已编译 Kernel 任务组织为一个超级 Kernel
- C. 强制模型只在 CPU 上执行
- D. 删除所有同步操作

（2）【单选题】开启 SuperKernel 融合优化时，必须同时开启哪个功能？
- A. force_eager
- B. static_kernel_compile
- C. clone_input
- D. cache_compile

（3）【单选题】用于标定 SuperKernel 融合范围的一对接口是？
- A. `Event.record` 和 `Event.wait`
- B. `super_kernel_scope_begin` 和 `super_kernel_scope_end`
- C. `cache_compile` 和 `readable_cache`
- D. `record_stream` 和 `synchronize`

（4）【单选题】当 `super_kernel_scope_begin` 传入的 `scope_name` 为 `None` 时，表示什么？
- A. 该范围内算子不进行 SuperKernel 融合
- B. 强制融合整个模型
- C. 自动启用 Data Dump
- D. 开启所有调试选项

（5）【单选题】开启 SuperKernel 融合后，哪项功能将失效？
- A. 算子 Data Dump
- B. 模型前向计算
- C. 静态 Kernel 编译
- D. NPU Profiler

（6）【单选题】`super_kernel_debug_options` 的推荐使用场景是？
- A. 始终用于生产性能测试
- B. 功能调试和问题定界
- C. 替代精度校验
- D. 替代静态 Kernel 编译

### 二、多选题

（7）【多选题】关于 SuperKernel 的描述，正确的有哪些？
- A. 可减少任务下发、调度等待和算子头部开销
- B. 融合结果受算子依赖、同步行为和资源占用影响
- C. 需要使用 Profiling 和精度结果评估收益
- D. 所有模型开启后都必然加速

（8）【多选题】关于 SuperKernel 范围标定，正确的说法有哪些？
- A. scope_name 用于标识标定范围融合后的 SuperKernel 名称
- B. 标定范围不代表最终一定会融合成功
- C. 标记为 `None` 的范围具有最高优先级且不参与融合
- D. scope_name 可以无限长且数量无限制

（9）【多选题】下列哪些是 `super_kernel_optimize_options` 中介绍的优化参数？
- A. auto_op_parallel
- B. early_start
- C. aggressive_opt_strategies
- D. debug_op_exec_trace

（10）【多选题】定位 SuperKernel 问题或评估收益时，合理的做法有哪些？
- A. 使用 `debug_op_exec_trace` 等调试选项辅助定界
- B. 使用 Profiling 确认融合结果和性能表现
- C. 与未融合版本做数值精度对比
- D. 长期在生产环境开启全部调试选项

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/04.06_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
